# Cloud-Cover Regressor - Balloon ML Payload (v2, regression variant)

Alternative to the 3-class cloud classifier. Instead of predicting clear / partial / overcast,
this model predicts the **cloud-cover fraction directly** as a continuous value in [0, 1]. It is
trained and evaluated on the same 95-Cloud-derived data, so the two can be compared head to head.

**Output:** `cloud_regressor.keras` (float). The classifier notebook is left untouched; this is a
parallel experiment you can keep or discard.

## 1. Approach

**Why regression.** The ground truth is a continuous cloud-pixel fraction (computed from the
segmentation masks by `build_cloud_classes.py` and stored in `manifest.csv`). The classifier threw
that number away by binning it into three classes, which created the partial-class boundary problem:
a patch at 48% and one at 52% cloud are nearly identical but get opposite labels. Predicting the
fraction directly removes the arbitrary bin boundaries entirely. The output is also more informative
(a percentage rather than a coarse bucket), and the telemetry shrinks to a single scalar.

**Method.** Same MobileNetV2 transfer-learning setup as the classifier (Edge TPU compatible
backbone, two-phase frozen-then-fine-tuned training, grayscale input replicated to 3 channels,
brightness/contrast augmentation for illumination robustness). Only the head and loss change: a
single sigmoid output (bounded to [0, 1]) trained with mean-squared error, reported as mean absolute
error (MAE) in cloud fraction.

**How we compare to the classifier.** At evaluation the predicted fraction is binned with the same
METAR thresholds (clear < 12.5%, partial 12.5-87.5%, overcast > 87.5%) into the three classes, so a
direct 3-class accuracy and confusion matrix can be placed next to the classifier's 0.805. The
regression also reports MAE, which the classifier cannot.

## 2. Dataset

Same source as the classifier: 95-Cloud (additional-to-38-Cloud), Landsat 8, originally per-pixel
cloud segmentation. Labels here are the continuous `cloud_fraction` column from
`cloud_classes/manifest.csv` (cloud-pixel fraction over the valid area of each patch), rather than
the binned class. Image preprocessing is identical: grayscale to match the HM01B0 camera, NIR
dropped, margin patches already filtered at generation time (min-valid 0.99).

## 3. Mission assumptions and limitations

These carry over unchanged from the classifier; the output format does not affect them.

**3.1 Viewing geometry.** Trained on satellite nadir view; the flight camera must look down.
Oblique mounting is out of distribution.

**3.2 Altitude and scale gap (dominant validation risk).** Landsat images from ~700 km at 30 m/px;
the balloon flies at ~30 km. Cloud scale and parallax differ, and this sim-to-real gap can only be
validated against real captured frames (the SD-card logging exists for this).

**3.3 Spectral channels (mono vs RGB).** Trained on grayscale to match the monochrome camera. RGB
would give a modest gain, mostly on bright-surface disambiguation; the decisive cue (NIR/SWIR) is on
no consumer camera. The bright-surface ambiguity (snow/ice/desert read as cloud) is intrinsic to
grayscale and limits accuracy on bright scenes.

**3.4 Illumination and time of day.** Landsat 8 is sun-synchronous (~10:30 local crossing): all
training imagery is daytime, mid-morning sun angles. Brightness/contrast augmentation here partially
mitigates this. Valid for daytime flight with reasonable sun elevation; dawn/dusk degrade, night is
moot for a visible camera.

**3.5 Surface domain.** Global Landsat scenes include snow, ice, and desert; the mission flies over
temperate Bavarian farmland in daylight, so those confusers are largely absent from the real flight
domain.

## 4. Setup

In [ ]:
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TF", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "NONE (CPU - slower but works)")

DATA_DIR   = Path("cloud_classes")     # contains manifest.csv + class folders of PNGs
IMG_SIZE   = (224, 224)
BATCH      = 32
SEED       = 1337
EPOCHS_HEAD = 15
EPOCHS_FT   = 40
FINE_TUNE_AT = 80
MODEL_OUT  = "cloud_regressor.keras"
EDGES = [0.125, 0.875]                  # METAR thresholds for the binned comparison

## 5. Load continuous labels from the manifest, split 80/20

In [ ]:
recs = list(csv.DictReader(open(DATA_DIR / "manifest.csv")))
paths = [str(DATA_DIR / r["class"] / (r["patch"] + ".png")) for r in recs]
fracs = [float(r["cloud_fraction"]) for r in recs]
print("patches:", len(paths), "| fraction range: %.3f - %.3f" % (min(fracs), max(fracs)))

idx = np.arange(len(paths))
np.random.RandomState(SEED).shuffle(idx)
cut = int(0.8 * len(idx))
tr_i, va_i = idx[:cut], idx[cut:]
print("train:", len(tr_i), "val:", len(va_i))

## 6. Input pipeline (augment train only; label is the fraction)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load(path, frac):
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)   # grayscale
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)                               # [0,255]
    return img, frac

def augment(x, y):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    x = tf.image.random_brightness(x, max_delta=25.0)
    x = tf.image.random_contrast(x, 0.85, 1.15)
    x = tf.clip_by_value(x, 0.0, 255.0)
    return x, y

def to_model(x, y):
    x = tf.image.grayscale_to_rgb(x)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    return x, y

def make_ds(indices, training):
    p = [paths[i] for i in indices]
    f = [fracs[i] for i in indices]
    ds = tf.data.Dataset.from_tensor_slices((p, f)).map(load, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(2000).map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(to_model, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(tr_i, True)
val_ds   = make_ds(va_i, False)

## 7. Sanity check: one batch with its fraction labels

In [ ]:
xb, fb = next(iter(train_ds))
print("batch:", xb.shape, "| label range: %.2f - %.2f" % (float(fb.numpy().min()), float(fb.numpy().max())))
plt.figure(figsize=(9, 9))
for i in range(min(9, xb.shape[0])):
    plt.subplot(3, 3, i + 1)
    plt.imshow((xb[i].numpy() + 1) / 2)
    plt.title("%.0f%% cloud" % (100 * float(fb[i]))); plt.axis("off")
plt.tight_layout(); plt.show()

## 8. Model and Phase 1 (frozen backbone + regression head)

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)    # cloud fraction in [0,1]
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)

## 9. Phase 2 fine-tuning (unfreeze upper backbone, low LR)

In [ ]:
base.trainable = True
for layer in base.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="mse", metrics=["mae"])
early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=EPOCHS_FT, callbacks=[early])

## 10. Training curves (MAE)

In [ ]:
def plot_mae(h, title):
    plt.figure(figsize=(6,4))
    plt.plot(h["mae"], label="train"); plt.plot(h["val_mae"], label="val")
    plt.title(title + " MAE (cloud fraction)"); plt.xlabel("epoch"); plt.legend(); plt.show()
plot_mae(history.history, "phase 1")
plot_mae(history_ft.history, "fine-tune")

## 11. Save

In [ ]:
model.save(MODEL_OUT)
print("saved", MODEL_OUT)

## 12. Evaluation: MAE, scatter, and binned comparison to the classifier

The headline regression metric is MAE in cloud fraction. To compare against the classifier we bin
the predicted fraction with the METAR thresholds and compute 3-class accuracy on the same validation
patches. The scatter plot shows where the model is accurate and where error concentrates (expect the
mid-range to be hardest, the regression analogue of the partial class).

In [ ]:
y_true_f, y_pred_f = [], []
for xb, fb in val_ds:
    pr = model.predict(xb, verbose=0).ravel()
    y_pred_f.extend(pr.tolist())
    y_true_f.extend(fb.numpy().tolist())
y_true_f = np.array(y_true_f); y_pred_f = np.array(y_pred_f)

mae = np.mean(np.abs(y_true_f - y_pred_f))
print("Validation MAE: %.4f  (= %.1f percentage points of cloud cover)" % (mae, 100*mae))

# scatter
plt.figure(figsize=(5,5))
plt.scatter(y_true_f, y_pred_f, s=4, alpha=0.3)
plt.plot([0,1],[0,1],"r--"); plt.xlabel("true fraction"); plt.ylabel("predicted")
plt.title("predicted vs true cloud fraction"); plt.show()

# binned 3-class comparison (clear=0, overcast=1, partial=2 to match classifier order)
def to_class(f):
    return 0 if f < EDGES[0] else (1 if f >= EDGES[1] else 2)
names = ["clear","overcast","partial"]
ct = np.array([to_class(f) for f in y_true_f])
cp = np.array([to_class(f) for f in y_pred_f])
cm = tf.math.confusion_matrix(ct, cp, num_classes=3).numpy()
acc = np.trace(cm)/cm.sum()
print("\nBinned 3-class accuracy: %.3f   (classifier was 0.805)" % acc)
print("Confusion matrix, order:", names); print(cm)
for i,c in enumerate(names):
    s = cm[i].sum(); print("  %-9s recall %.3f" % (c, cm[i,i]/s if s else 0))

# MAE by true bin (where does error concentrate?)
print("\nMAE by true class:")
for i,c in enumerate(names):
    m = ct==i
    if m.any(): print("  %-9s %.3f (%.1f pp)" % (c, np.mean(np.abs(y_true_f[m]-y_pred_f[m])), 100*np.mean(np.abs(y_true_f[m]-y_pred_f[m]))))

## 13. Int8 quantization (PTQ) and int8 evaluation

In [ ]:
# representative set spanning the fraction range for good calibration
def representative_dataset():
    order = sorted(range(len(tr_i)), key=lambda k: fracs[tr_i[k]])
    pick = order[::max(1, len(order)//400)][:400]
    for k in pick:
        img = tf.io.decode_png(tf.io.read_file(paths[tr_i[k]]), channels=1)
        img = tf.image.resize(img, IMG_SIZE); img = tf.cast(img, tf.float32)
        img = tf.image.grayscale_to_rgb(img)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        yield [img[tf.newaxis, ...]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tflite_model = conv.convert()
open("cloud_regressor_int8.tflite","wb").write(tflite_model)
print("wrote cloud_regressor_int8.tflite,", len(tflite_model), "bytes")

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
ind, outd = interp.get_input_details()[0], interp.get_output_details()[0]
isc, izp = ind["quantization"]; osc, ozp = outd["quantization"]
print("input scale/zp:", isc, izp, "| output scale/zp:", osc, ozp)   # C++ needs these

yt, yp = [], []
for xb, fb in val_ds:
    for img, fr in zip(xb.numpy(), fb.numpy()):
        q = np.round(img/isc + izp).clip(-128,127).astype(np.int8)
        interp.set_tensor(ind["index"], q[np.newaxis,...]); interp.invoke()
        raw = interp.get_tensor(outd["index"])[0][0]
        yp.append((float(raw) - ozp) * osc)        # dequantize -> fraction
        yt.append(float(fr))
yt, yp = np.array(yt), np.array(yp)
print("\nINT8 MAE: %.4f (%.1f pp)" % (np.mean(np.abs(yt-yp)), 100*np.mean(np.abs(yt-yp))))
ct = np.array([to_class(f) for f in yt]); cp = np.array([to_class(f) for f in yp])
cm = tf.math.confusion_matrix(ct, cp, num_classes=3).numpy()
print("INT8 binned 3-class accuracy: %.3f   (classifier int8 was 0.805)" % (np.trace(cm)/cm.sum()))
print(cm)

## 14. Comparison summary

Fill in after running, to decide classifier vs regressor:

| model            | int8 binned 3-class acc | int8 MAE (pp) | partial/mid handling |
|------------------|-------------------------|---------------|----------------------|
| classifier (v1)  | 0.805                   | n/a           | partial recall 0.60  |
| regressor (v2)   | ?                       | ?             | mid-range MAE ?      |

If the regressor's binned accuracy matches or beats 0.805 **and** its MAE is low, it is the better
model: same or better classification, plus a continuous output and no bin-boundary fragility. If
binned accuracy is clearly worse, keep the classifier. The MAE-by-class numbers from Section 12 show
whether the mid-range (the old partial problem) is handled more gracefully as regression error than
as a collapsed class.